# Day 1 â€” PySpark: DataFrame Filtering
### filter(), where(), isin(), between(), isNull(), HAVING equivalent
### Target: Data Engineer Interviews

> **Roadmap Day:** 1 Â· **Date:** Monday, June 15, 2026  
> **Prerequisite:** Run `00_prerequisites.ipynb` first â€” Java and PySpark must be installed.  
> **Same problems as SQL Day 1 â€” same dataset, PySpark API.**

---
## Setup â€” SparkSession

In [1]:
import os, sys

print("Environment set. Run next cell to start Spark.")

Environment set. Run next cell to start Spark.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, avg, count, min as spark_min, max as spark_max,
    round as spark_round, year, to_date, lower
)
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

spark = SparkSession.builder \
    .appName("Day01_Filtering") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready.")

Spark 4.1.1 ready.


---
## 1. The employees DataFrame

Mirrors the `employees` table from `setup_tables.sql` exactly â€” same data, same columns.

In [3]:
# Same data as setup_tables.sql employees table
data = [
    (1,  "Alice Johnson",  "alice@company.com",   "Engineering",  95000.0, 8000.0,  None, "2019-03-15", "active"),
    (2,  "Bob Smith",      "bob@company.com",     "Engineering",  65000.0, 3000.0,  1,    "2021-07-01", "active"),
    (3,  "Carol White",    "carol@company.com",   "Finance",      85000.0, 7000.0,  None, "2018-01-10", "active"),
    (4,  "Dave Brown",     "dave@company.com",    "Finance",      60000.0, 2500.0,  3,    "2022-04-20", "active"),
    (5,  "Eve Davis",      "eve@company.com",     "HR",           75000.0, 5000.0,  None, "2017-06-05", "active"),
    (6,  "Frank Miller",   "frank@company.com",   "HR",           50000.0, 1500.0,  5,    "2023-01-15", "active"),
    (7,  "Grace Wilson",   "grace@company.com",   "Marketing",    80000.0, 6000.0,  None, "2020-09-01", "active"),
    (8,  "Henry Moore",    "henry@company.com",   "Marketing",    55000.0, 2000.0,  7,    "2022-11-10", "inactive"),
    (9,  "Iris Taylor",    "iris@company.com",    "Engineering", 110000.0,10000.0,  1,    "2016-02-28", "active"),
    (10, "Jack Anderson",  "jack@company.com",    "Operations",   72000.0, 4000.0,  None, "2019-08-12", "active"),
    (11, "Karen Thomas",   "karen@company.com",   "Operations",   48000.0, 1000.0,  10,   "2023-03-01", "active"),
    (12, "Leo Jackson",    "leo@company.com",     "Engineering",  78000.0, 5500.0,  1,    "2020-05-17", "active"),
    (13, "Mia Harris",     "mia@company.com",     "Finance",      70000.0, 4500.0,  3,    "2021-02-28", "active"),
    (14, "Nathan Clark",   "nathan@company.com",  "Marketing",    45000.0,  800.0,  7,    "2023-06-01", "active"),
    (15, "Olivia Lewis",   "olivia@company.com",  "HR",           52000.0, 1800.0,  5,    "2022-08-15", "active"),
]

schema = StructType([
    StructField("id",         IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("email",      StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("salary",     FloatType(),   True),
    StructField("bonus",      FloatType(),   True),
    StructField("manager_id", IntegerType(), True),
    StructField("hire_date",  StringType(),  True),
    StructField("status",     StringType(),  True),
])

df = spark.createDataFrame(data, schema=schema)

# Cast hire_date to DateType for proper date operations
df = df.withColumn("hire_date", to_date(col("hire_date"), "yyyy-MM-dd"))

df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: float (nullable = true)
 |-- bonus: float (nullable = true)
 |-- manager_id: integer (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- status: string (nullable = true)



In [4]:
df.select("id", "name", "department", "salary", "hire_date", "status").show(truncate=False)

+---+-------------+-----------+--------+----------+--------+
|id |name         |department |salary  |hire_date |status  |
+---+-------------+-----------+--------+----------+--------+
|1  |Alice Johnson|Engineering|95000.0 |2019-03-15|active  |
|2  |Bob Smith    |Engineering|65000.0 |2021-07-01|active  |
|3  |Carol White  |Finance    |85000.0 |2018-01-10|active  |
|4  |Dave Brown   |Finance    |60000.0 |2022-04-20|active  |
|5  |Eve Davis    |HR         |75000.0 |2017-06-05|active  |
|6  |Frank Miller |HR         |50000.0 |2023-01-15|active  |
|7  |Grace Wilson |Marketing  |80000.0 |2020-09-01|active  |
|8  |Henry Moore  |Marketing  |55000.0 |2022-11-10|inactive|
|9  |Iris Taylor  |Engineering|110000.0|2016-02-28|active  |
|10 |Jack Anderson|Operations |72000.0 |2019-08-12|active  |
|11 |Karen Thomas |Operations |48000.0 |2023-03-01|active  |
|12 |Leo Jackson  |Engineering|78000.0 |2020-05-17|active  |
|13 |Mia Harris   |Finance    |70000.0 |2021-02-28|active  |
|14 |Nathan Clark |Marke

---
## 2. filter() and where() â€” Two Names, One Function

`filter()` and `where()` are **identical aliases** â€” same behavior, interchangeable.  
Both accept a Column expression or a SQL string.

In [5]:
# Column expression â€” recommended for production code
df.filter(col("department") == "Engineering") \
  .select("id", "name", "department", "salary") \
  .orderBy(col("salary").desc()) \
  .show()

+---+-------------+-----------+--------+
| id|         name| department|  salary|
+---+-------------+-----------+--------+
|  9|  Iris Taylor|Engineering|110000.0|
|  1|Alice Johnson|Engineering| 95000.0|
| 12|  Leo Jackson|Engineering| 78000.0|
|  2|    Bob Smith|Engineering| 65000.0|
+---+-------------+-----------+--------+



In [6]:
# where() â€” same result
df.where("department = 'Engineering'") \
  .select("id", "name", "department", "salary") \
  .orderBy(col("salary").desc()) \
  .show()

+---+-------------+-----------+--------+
| id|         name| department|  salary|
+---+-------------+-----------+--------+
|  9|  Iris Taylor|Engineering|110000.0|
|  1|Alice Johnson|Engineering| 95000.0|
| 12|  Leo Jackson|Engineering| 78000.0|
|  2|    Bob Smith|Engineering| 65000.0|
+---+-------------+-----------+--------+



In [10]:
# Three equivalent syntaxes â€” all return the same rows
r1 = df.filter(col("department") == "Engineering")
r2 = df.filter(df["department"] == "Engineering")
r3 = df.filter("department = 'Engineering'")

print("All three produce the same result:",
      r1.count() == r2.count() == r3.count() == True)
print(r1.count(), r2.count(), r3.count())

All three produce the same result: False
4 4 4


---
## 3. between() â€” Inclusive Range Filter

Equivalent to SQL `BETWEEN low AND high` â€” inclusive on both ends.

In [11]:
# Numeric range â€” salary 60000â€“100000 inclusive
df.filter(col("salary").between(60000, 100000)) \
  .select("id", "name", "department", "salary") \
  .orderBy("salary") \
  .show()

+---+-------------+-----------+-------+
| id|         name| department| salary|
+---+-------------+-----------+-------+
|  4|   Dave Brown|    Finance|60000.0|
|  2|    Bob Smith|Engineering|65000.0|
| 13|   Mia Harris|    Finance|70000.0|
| 10|Jack Anderson| Operations|72000.0|
|  5|    Eve Davis|         HR|75000.0|
| 12|  Leo Jackson|Engineering|78000.0|
|  7| Grace Wilson|  Marketing|80000.0|
|  3|  Carol White|    Finance|85000.0|
|  1|Alice Johnson|Engineering|95000.0|
+---+-------------+-----------+-------+



In [12]:
# Date range â€” hired in 2022
df.filter(col("hire_date").between("2022-01-01", "2022-12-31")) \
  .select("id", "name", "hire_date") \
  .orderBy("hire_date") \
  .show()

+---+------------+----------+
| id|        name| hire_date|
+---+------------+----------+
|  4|  Dave Brown|2022-04-20|
| 15|Olivia Lewis|2022-08-15|
|  8| Henry Moore|2022-11-10|
+---+------------+----------+



In [13]:
# Alternative: year() function for year-based filtering
df.filter(year(col("hire_date")) == 2022) \
  .select("id", "name", "hire_date") \
  .show()

+---+------------+----------+
| id|        name| hire_date|
+---+------------+----------+
|  4|  Dave Brown|2022-04-20|
|  8| Henry Moore|2022-11-10|
| 15|Olivia Lewis|2022-08-15|
+---+------------+----------+



---
## 4. isin() â€” Match Any Value in a List

In [14]:
# isin â€” equivalent to SQL IN (...)
df.filter(col("department").isin("Engineering", "Finance")) \
  .select("id", "name", "department") \
  .orderBy("department", "name") \
  .show()

+---+-------------+-----------+
| id|         name| department|
+---+-------------+-----------+
|  1|Alice Johnson|Engineering|
|  2|    Bob Smith|Engineering|
|  9|  Iris Taylor|Engineering|
| 12|  Leo Jackson|Engineering|
|  3|  Carol White|    Finance|
|  4|   Dave Brown|    Finance|
| 13|   Mia Harris|    Finance|
+---+-------------+-----------+



In [15]:
# isin with a Python list variable
target_depts = ["Engineering", "Finance"]

df.filter(col("department").isin(target_depts)) \
  .select("name", "department") \
  .show()

+-------------+-----------+
|         name| department|
+-------------+-----------+
|Alice Johnson|Engineering|
|    Bob Smith|Engineering|
|  Carol White|    Finance|
|   Dave Brown|    Finance|
|  Iris Taylor|Engineering|
|  Leo Jackson|Engineering|
|   Mia Harris|    Finance|
+-------------+-----------+



In [16]:
# NOT IN â€” use ~ (tilde) to negate
df.filter(~col("department").isin("HR", "Marketing")) \
  .select("name", "department") \
  .orderBy("department", "name") \
  .show()

+-------------+-----------+
|         name| department|
+-------------+-----------+
|Alice Johnson|Engineering|
|    Bob Smith|Engineering|
|  Iris Taylor|Engineering|
|  Leo Jackson|Engineering|
|  Carol White|    Finance|
|   Dave Brown|    Finance|
|   Mia Harris|    Finance|
|Jack Anderson| Operations|
| Karen Thomas| Operations|
+-------------+-----------+



---
## 5. String Pattern Matching

PySpark has dedicated string methods that mirror SQL `LIKE` and go beyond it.

In [17]:
# startswith â€” SQL LIKE 'A%'
df.filter(col("name").startswith("A")) \
  .select("id", "name") \
  .show()

+---+-------------+
| id|         name|
+---+-------------+
|  1|Alice Johnson|
+---+-------------+



In [18]:
# endswith â€” SQL LIKE '%son'
df.filter(col("name").endswith("son")) \
  .select("name") \
  .show()

+-------------+
|         name|
+-------------+
|Alice Johnson|
| Grace Wilson|
|Jack Anderson|
|  Leo Jackson|
+-------------+



In [19]:
# contains â€” SQL LIKE '%ar%'
df.filter(col("name").contains("ar")) \
  .select("name") \
  .show()

+------------+
|        name|
+------------+
| Carol White|
|Karen Thomas|
|  Mia Harris|
|Nathan Clark|
+------------+



In [20]:
# like() â€” same SQL % wildcard syntax
df.filter(col("name").like("A%")) \
  .select("name") \
  .show()

+-------------+
|         name|
+-------------+
|Alice Johnson|
+-------------+



In [21]:
# rlike() â€” full regex (much more powerful than SQL LIKE)
# Match names starting with 'A' or 'I'
df.filter(col("name").rlike("^[AI]")) \
  .select("name") \
  .show()

+-------------+
|         name|
+-------------+
|Alice Johnson|
|  Iris Taylor|
+-------------+



In [22]:
# Case-insensitive â€” use lower()
# SQL ILIKE 'a%' equivalent
df.filter(lower(col("name")).startswith("a")) \
  .select("name") \
  .show()

+-------------+
|         name|
+-------------+
|Alice Johnson|
+-------------+



In [23]:
# Email domain filter â€” gmail users
df.filter(col("email").endswith("@company.com")) \
  .select("name", "email") \
  .show()

+-------------+------------------+
|         name|             email|
+-------------+------------------+
|Alice Johnson| alice@company.com|
|    Bob Smith|   bob@company.com|
|  Carol White| carol@company.com|
|   Dave Brown|  dave@company.com|
|    Eve Davis|   eve@company.com|
| Frank Miller| frank@company.com|
| Grace Wilson| grace@company.com|
|  Henry Moore| henry@company.com|
|  Iris Taylor|  iris@company.com|
|Jack Anderson|  jack@company.com|
| Karen Thomas| karen@company.com|
|  Leo Jackson|   leo@company.com|
|   Mia Harris|   mia@company.com|
| Nathan Clark|nathan@company.com|
| Olivia Lewis|olivia@company.com|
+-------------+------------------+



---
## 6. isNull() and isNotNull()

In [24]:
# isNull â€” employees with no manager (top-level)
df.filter(col("manager_id").isNull()) \
  .select("id", "name", "department", "manager_id") \
  .orderBy("department") \
  .show()

+---+-------------+-----------+----------+
| id|         name| department|manager_id|
+---+-------------+-----------+----------+
|  1|Alice Johnson|Engineering|      NULL|
|  3|  Carol White|    Finance|      NULL|
|  5|    Eve Davis|         HR|      NULL|
|  7| Grace Wilson|  Marketing|      NULL|
| 10|Jack Anderson| Operations|      NULL|
+---+-------------+-----------+----------+



In [25]:
# isNotNull â€” employees who report to someone
df.filter(col("manager_id").isNotNull()) \
  .select("id", "name", "department", "manager_id") \
  .orderBy("manager_id", "name") \
  .show()

+---+------------+-----------+----------+
| id|        name| department|manager_id|
+---+------------+-----------+----------+
|  2|   Bob Smith|Engineering|         1|
|  9| Iris Taylor|Engineering|         1|
| 12| Leo Jackson|Engineering|         1|
|  4|  Dave Brown|    Finance|         3|
| 13|  Mia Harris|    Finance|         3|
|  6|Frank Miller|         HR|         5|
| 15|Olivia Lewis|         HR|         5|
|  8| Henry Moore|  Marketing|         7|
| 14|Nathan Clark|  Marketing|         7|
| 11|Karen Thomas| Operations|        10|
+---+------------+-----------+----------+



In [26]:
# Gotcha: comparing to None does NOT match null rows
df_test = spark.createDataFrame([(1, None), (2, 100.0)], ["id", "val"])

# Wrong â€” returns 0 rows
wrong_count = df_test.filter(col("val") == None).count()
print(f"col == None: {wrong_count} rows (wrong approach)")

# Correct â€” returns 1 row
right_count = df_test.filter(col("val").isNull()).count()
print(f"col.isNull(): {right_count} row (correct approach)")

col == None: 0 rows (wrong approach)
col.isNull(): 1 row (correct approach)


---
## 7. Combining Conditions â€” &, |, ~

> **Critical:** Every condition must be in `()` parentheses when combining with `&` or `|`.  
> Python's operator precedence causes silent, hard-to-debug bugs without them.

In [27]:
# AND â€” use & with parentheses around each condition
df.filter(
    (col("department") == "Engineering") &
    (col("salary").between(60000, 100000))
).select("id", "name", "department", "salary") \
 .orderBy(col("salary").desc()) \
 .show()

+---+-------------+-----------+-------+
| id|         name| department| salary|
+---+-------------+-----------+-------+
|  1|Alice Johnson|Engineering|95000.0|
| 12|  Leo Jackson|Engineering|78000.0|
|  2|    Bob Smith|Engineering|65000.0|
+---+-------------+-----------+-------+



In [28]:
# OR â€” use |
df.filter(
    col("name").startswith("A") |
    (year(col("hire_date")) == 2022)
).select("id", "name", "department", "hire_date") \
 .orderBy("hire_date") \
 .show()

+---+-------------+-----------+----------+
| id|         name| department| hire_date|
+---+-------------+-----------+----------+
|  1|Alice Johnson|Engineering|2019-03-15|
|  4|   Dave Brown|    Finance|2022-04-20|
| 15| Olivia Lewis|         HR|2022-08-15|
|  8|  Henry Moore|  Marketing|2022-11-10|
+---+-------------+-----------+----------+



In [29]:
# NOT â€” use ~
df.filter(~col("department").isin("HR", "Marketing")) \
  .select("name", "department") \
  .orderBy("department", "name") \
  .show()

+-------------+-----------+
|         name| department|
+-------------+-----------+
|Alice Johnson|Engineering|
|    Bob Smith|Engineering|
|  Iris Taylor|Engineering|
|  Leo Jackson|Engineering|
|  Carol White|    Finance|
|   Dave Brown|    Finance|
|   Mia Harris|    Finance|
|Jack Anderson| Operations|
| Karen Thomas| Operations|
+-------------+-----------+



In [30]:
# Three conditions chained
df.filter(
    (col("status") == "active") &
    (col("salary") > 70000) &
    (col("department").isin("Engineering", "Finance"))
).select("name", "department", "salary", "status") \
 .orderBy(col("salary").desc()) \
 .show()

+-------------+-----------+--------+------+
|         name| department|  salary|status|
+-------------+-----------+--------+------+
|  Iris Taylor|Engineering|110000.0|active|
|Alice Johnson|Engineering| 95000.0|active|
|  Carol White|    Finance| 85000.0|active|
|  Leo Jackson|Engineering| 78000.0|active|
+-------------+-----------+--------+------+



In [31]:
# Also: use & | ~ NOT Python 'and' 'or' 'not'
try:
    df.filter(col("department") == "Engineering" and col("salary") > 80000).show()
except Exception as e:
    print(f"Using 'and' fails: {type(e).__name__}: {e}")
    print("Use & instead of 'and', | instead of 'or', ~ instead of 'not'")

Using 'and' fails: PySparkValueError: [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.
Use & instead of 'and', | instead of 'or', ~ instead of 'not'


In [32]:
# SQL: GROUP BY department HAVING AVG(salary) > 70000

df.groupBy("department") \
  .agg(
      spark_round(avg("salary"), 2).alias("avg_salary"),
      count("*").alias("headcount")
  ) \
  .filter(col("avg_salary") > 70000) \
  .orderBy(col("avg_salary").desc()) \
  .show()

+-----------+----------+---------+
| department|avg_salary|headcount|
+-----------+----------+---------+
|Engineering|   87000.0|        4|
|    Finance|  71666.67|        3|
+-----------+----------+---------+



In [33]:
# All departments â€” comparison
df.groupBy("department") \
  .agg(
      spark_round(avg("salary"), 0).alias("avg_salary"),
      count("*").alias("headcount"),
      spark_min("salary").alias("min_sal"),
      spark_max("salary").alias("max_sal")
  ) \
  .orderBy(col("avg_salary").desc()) \
  .show()

+-----------+----------+---------+-------+--------+
| department|avg_salary|headcount|min_sal| max_sal|
+-----------+----------+---------+-------+--------+
|Engineering|   87000.0|        4|65000.0|110000.0|
|    Finance|   71667.0|        3|60000.0| 85000.0|
|  Marketing|   60000.0|        3|45000.0| 80000.0|
| Operations|   60000.0|        2|48000.0| 72000.0|
|         HR|   59000.0|        3|50000.0| 75000.0|
+-----------+----------+---------+-------+--------+



In [34]:
# WHERE + HAVING equivalent
# Active employees (row filter) â†’ departments with 2+ people and avg > 60000 (group filter)
df.filter(col("status") == "active") \
  .groupBy("department") \
  .agg(
      count("*").alias("headcount"),
      spark_round(avg("salary"), 0).alias("avg_salary")
  ) \
  .filter(
      (col("headcount") >= 2) &
      (col("avg_salary") > 60000)
  ) \
  .orderBy(col("avg_salary").desc()) \
  .show()

+-----------+---------+----------+
| department|headcount|avg_salary|
+-----------+---------+----------+
|Engineering|        4|   87000.0|
|    Finance|        3|   71667.0|
|  Marketing|        2|   62500.0|
+-----------+---------+----------+



---
## 9. Day 1 Problems â€” Official Solutions

Try writing your solution before running each cell.

In [35]:
# Problem 1 (Easy)
# Find all employees in the 'Engineering' department with salary between 60000 and 100000

print("Problem 1 â€” Engineering dept, salary 60kâ€“100k")
print("=" * 52)

df.filter(
    (col("department") == "Engineering") &
    (col("salary").between(60000, 100000))
).select("id", "name", "department", "salary") \
 .orderBy(col("salary").desc()) \
 .show()

# Expected: Alice 95000, Leo 78000, Bob 65000
# Iris 110000 â†’ excluded (above 100000)

Problem 1 â€” Engineering dept, salary 60kâ€“100k
+---+-------------+-----------+-------+
| id|         name| department| salary|
+---+-------------+-----------+-------+
|  1|Alice Johnson|Engineering|95000.0|
| 12|  Leo Jackson|Engineering|78000.0|
|  2|    Bob Smith|Engineering|65000.0|
+---+-------------+-----------+-------+



In [36]:
# Problem 2 (Easy)
# Find departments where the average salary is greater than 70000

print("Problem 2 â€” Departments with avg salary > 70000")
print("=" * 52)

df.groupBy("department") \
  .agg(
      spark_round(avg("salary"), 2).alias("avg_salary"),
      count("*").alias("headcount")
  ) \
  .filter(col("avg_salary") > 70000) \
  .orderBy(col("avg_salary").desc()) \
  .show()

Problem 2 â€” Departments with avg salary > 70000
+-----------+----------+---------+
| department|avg_salary|headcount|
+-----------+----------+---------+
|Engineering|   87000.0|        4|
|    Finance|  71666.67|        3|
+-----------+----------+---------+



In [37]:
# Problem 3 (Medium)
# Find employees whose name starts with 'A' OR hired in 2022

print("Problem 3 â€” Name starts with A OR hired in 2022")
print("=" * 52)

df.filter(
    col("name").startswith("A") |
    (year(col("hire_date")) == 2022)
).select("id", "name", "department", "salary", "hire_date") \
 .orderBy("hire_date") \
 .show()

Problem 3 â€” Name starts with A OR hired in 2022
+---+-------------+-----------+-------+----------+
| id|         name| department| salary| hire_date|
+---+-------------+-----------+-------+----------+
|  1|Alice Johnson|Engineering|95000.0|2019-03-15|
|  4|   Dave Brown|    Finance|60000.0|2022-04-20|
| 15| Olivia Lewis|         HR|52000.0|2022-08-15|
|  8|  Henry Moore|  Marketing|55000.0|2022-11-10|
+---+-------------+-----------+-------+----------+



---
## 10. SparkSQL Alternative â€” Same Queries via SQL String

In [38]:
# Register as temp view â€” then use SQL directly
df.createOrReplaceTempView("employees")

# Problem 1 via SparkSQL
spark.sql("""
    SELECT id, name, department, salary
    FROM employees
    WHERE department = 'Engineering'
      AND salary BETWEEN 60000 AND 100000
    ORDER BY salary DESC
""").show()

+---+-------------+-----------+-------+
| id|         name| department| salary|
+---+-------------+-----------+-------+
|  1|Alice Johnson|Engineering|95000.0|
| 12|  Leo Jackson|Engineering|78000.0|
|  2|    Bob Smith|Engineering|65000.0|
+---+-------------+-----------+-------+



In [39]:
# Problem 2 via SparkSQL
spark.sql("""
    SELECT department,
           ROUND(AVG(salary), 2) AS avg_salary,
           COUNT(*) AS headcount
    FROM employees
    GROUP BY department
    HAVING AVG(salary) > 70000
    ORDER BY avg_salary DESC
""").show()

+-----------+----------+---------+
| department|avg_salary|headcount|
+-----------+----------+---------+
|Engineering|   87000.0|        4|
|    Finance|  71666.67|        3|
+-----------+----------+---------+



In [40]:
# Problem 3 via SparkSQL â€” YEAR() function works in Spark SQL
spark.sql("""
    SELECT id, name, department, salary, hire_date
    FROM employees
    WHERE name LIKE 'A%'
       OR YEAR(hire_date) = 2022
    ORDER BY hire_date
""").show()

+---+-------------+-----------+-------+----------+
| id|         name| department| salary| hire_date|
+---+-------------+-----------+-------+----------+
|  1|Alice Johnson|Engineering|95000.0|2019-03-15|
|  4|   Dave Brown|    Finance|60000.0|2022-04-20|
| 15| Olivia Lewis|         HR|52000.0|2022-08-15|
|  8|  Henry Moore|  Marketing|55000.0|2022-11-10|
+---+-------------+-----------+-------+----------+



---
## 11. PySpark vs SQL â€” Full Mapping Table

| SQL | PySpark |
|-----|---------|
| `WHERE dept = 'Engineering'` | `filter(col("dept") == "Engineering")` |
| `WHERE salary BETWEEN 60000 AND 100000` | `filter(col("salary").between(60000, 100000))` |
| `WHERE dept IN ('Eng', 'Fin')` | `filter(col("dept").isin("Eng", "Fin"))` |
| `WHERE dept NOT IN ('HR')` | `filter(~col("dept").isin("HR"))` |
| `WHERE name LIKE 'A%'` | `filter(col("name").startswith("A"))` |
| `WHERE name ILIKE 'a%'` | `filter(lower(col("name")).startswith("a"))` |
| `WHERE salary IS NULL` | `filter(col("salary").isNull())` |
| `WHERE salary IS NOT NULL` | `filter(col("salary").isNotNull())` |
| `WHERE a AND b` | `filter((col_a) & (col_b))` |
| `WHERE a OR b` | `filter((col_a) \| (col_b))` |
| `WHERE NOT a` | `filter(~col_a)` |
| `HAVING AVG(sal) > 70000` | `.groupBy().agg(avg().alias("x")).filter(col("x") > 70000)` |

In [41]:
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
